# Step 6.b — Climate and soil suitability under the AMOC scenarios, global (EC-Earth3 grid)

The final rule of 5.h applied to ERA5 and to the four EC-Earth3 runs (CO2 fixed at the 2025 / 2050 level, without and
with the freshwater hosing that collapses the AMOC; delta BIO variables of 1.3.a, SVM of 3.g, climate probability of
4.g): **suitable = climate suitable (`suit_bin` of 4.g: probability ≥ `cu.CLIMATE_P` = 0.5 at equal priors and
growing-period temperature ≥ the FAO ECOCROP TMIN, with the temperatures of the scenario) AND soil rating ≥
`cu.SOIL_MIN` (25, GAEZ "moderate" or better)**, per crop and GAEZ input level (low, high). The soil is the same in every scenario, so a
difference between two scenarios comes from the climate only, on the land whose soil is suitable.

- **Novel climate** (4.g: MESS < 0, NT2 > 1 or no support vector within the reach of the SVM kernel): not suitable, and
  reported apart. The hosing effect is given on all land and on the cells assessable in both scenarios of the pair
  (novel in neither), as in 4.g.
- **Today's crop area**: the share of the MIRCA2000 harvested area of the crop in suitable cells: does the land where
  the crop grows today stay suitable?
- **Regions**: Europe (35–72° N, 25° W–45° E, the region of the RCM and LISFLOOD branch), north of 50° N, 23.5–50° N,
  the tropics and the southern extratropics. The AMOC collapse cools the northern hemisphere most (North Atlantic,
  Europe) and shifts the tropical rain belt south.

Outputs in `<out_root>/6_climate_soil_amoc/<route>/<variant>_<radiation>/`: `climate_soil_scenarios_<crop>.nc`
(`suitable_<level>`, `novel`, dimension `scenario`), `areas_scenarios_<set>.csv`, `amoc_effect_<set>.csv`,
`amoc_effect_regions_<set>.csv`, and the maps of section 3 in `<crop>/`.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the set of 4.g and 5.h
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # as 4.g and 5.h
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]           # GAEZ input level of the soil rating (2.g)

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
OUT = cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET)
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
LAND_MHA = LAND_HA * LAND / 1e6   # land area of the cells (Mha)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
print(f"set {SET} | rule: climate p >= {cu.CLIMATE_P} and >= ECOCROP TMIN, soil >= {cu.SOIL_MIN} | climate {CLIM_DIR} | outputs {OUT}")

## 1. Suitability per scenario

In [ ]:
mha = lambda a: float(np.nansum(LAND_MHA * a))
pct = lambda a, b: 100 * (a - b) / b if b else np.nan
area_rows, effect_rows, region_rows, MAPS = [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        missing = [s for s in SCEN if s not in ds.scenario.values]
        assert not missing, f"{crop}: scenarios {missing} not in 4.g"
        clim_ok = ds["suit_bin"].sel(scenario=SCEN).values == 1               # (scenario, lat, lon): p >= 0.5, >= TMIN, not novel
        novel = ds["novel"].sel(scenario=SCEN).values == 1
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    dv, suit = {}, {}
    for lv in LEVELS:
        ssr = np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0)      # 0-100, no soil rating -> 0 (5.h)
        suit[lv] = LAND & cu.suitable(clim_ok, ssr)                           # (scenario, lat, lon)
        dv[f"suitable_{lv}"] = (("scenario", "lat", "lon"), np.where(LAND, suit[lv], np.nan).astype("float32"),
                                {"long_name": f"final suitability, {lv} input: climate suitable (p >= {cu.CLIMATE_P}, >= ECOCROP TMIN) and soil >= {cu.SOIL_MIN}"})
    dv["novel"] = (("scenario", "lat", "lon"), novel.astype("int8"), {"long_name": "novel climate (4.g)"})
    xr.Dataset(dv, coords={"scenario": SCEN, "lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "set": SET, "route": ROUTE, "climate_p": cu.CLIMATE_P, "soil_min": cu.SOIL_MIN}
               ).to_netcdf(OUT / f"climate_soil_scenarios_{crop}.nc")
    MAPS[crop] = {"suit": suit, "novel": novel}

    for k, s in enumerate(SCEN):
        r = {"crop": crop, "scenario": s, "novel (Mha)": mha(novel[k]),
             "MIRCA area of the crop in novel cells (%)": 100 * crop_ha[novel[k]].sum() / crop_ha.sum()}
        for lv in LEVELS:
            r[f"suitable, {lv} (Mha)"] = mha(suit[lv][k])
            r[f"today's crop area suitable, {lv} (%)"] = 100 * crop_ha[suit[lv][k]].sum() / crop_ha.sum()
        area_rows.append(r)

    for hos, orig in PAIRS:
        h, o = SCEN.index(hos), SCEN.index(orig)
        both = LAND & ~(novel[h] | novel[o])
        for lv in LEVELS:
            sh, so = suit[lv][h], suit[lv][o]
            r = {"crop": crop, "pair": f"{hos} - {orig}", "level": lv, "novel in either (Mha)": mha(LAND & ~both),
                 "suitable change, all land (%)": pct(mha(sh), mha(so)),
                 "suitable change, assessable in both (%)": pct(mha(sh & both), mha(so & both)),
                 "gained (Mha)": mha(sh & ~so & both), "lost (Mha)": mha(so & ~sh & both),
                 "today's crop area suitable: change (points)": 100 * (crop_ha[sh].sum() - crop_ha[so].sum()) / crop_ha.sum()}
            effect_rows.append(r)
            for reg, rm in REGIONS.items():
                ao, ah = mha(so & both & rm), mha(sh & both & rm)
                region_rows.append({"crop": crop, "pair": f"{hos} - {orig}", "level": lv, "region": reg,
                                    "suitable, original (Mha)": ao, "suitable change (Mha)": ah - ao,
                                    "suitable change (%)": pct(ah, ao) if ao > 0.5 else np.nan})
    print(f"{crop}: done", flush=True)

## 2. Results

(1) Suitable land and share of today's crop area suitable, per scenario. (2) Effect of the AMOC collapse
(hosing − original): suitable land on all land and on the cells assessable in both, land gained and lost, change of the
share of today's crop area suitable. (3) The change by region (cells assessable in both). Consistency check: the
climate-only effect is in `amoc_effect_<set>.csv` of 4.g.

In [ ]:
areas = pd.DataFrame(area_rows).set_index(["crop", "scenario"])
effect = pd.DataFrame(effect_rows).set_index(["crop", "pair", "level"])
regions = pd.DataFrame(region_rows).set_index(["crop", "pair", "level", "region"])
areas.to_csv(OUT / f"areas_scenarios_{SET}.csv"); effect.to_csv(OUT / f"amoc_effect_{SET}.csv")
regions.to_csv(OUT / f"amoc_effect_regions_{SET}.csv")
with pd.option_context("display.max_rows", 300, "display.max_columns", 40, "display.width", 250):
    print("(1) suitable land per scenario (Mha) and share of today's crop area suitable (%)")
    display(areas.round(2))
    print("(2) effect of the AMOC collapse")
    display(effect.round(2))
    for lv in LEVELS:
        print(f"(3) suitable land change (%) by region, cells assessable in both, {lv} input")
        display(regions.xs(lv, level="level").reset_index().pivot_table(index=["crop", "pair"], columns="region", sort=False,
                values="suitable change (%)")[list(REGIONS)].round(1))

## 3. Final binary maps: ERA5 and the four runs

The final product per crop and input level (low, high): suitable (climate p >= 0.5 and >= ECOCROP TMIN, soil >= 25) / not suitable /
novel climate, the same rule for the five climates. Written in `<outputs>/<crop>/`, six PNG per input level:
0. `<crop>_<level>_0_ERA5_vs_MIRCA.png`: ERA5 against today's crop: suitable and cultivated (the model and MIRCA2000
   agree), cultivated but not suitable (missed by the model), suitable but not cultivated (potential land);
   cultivated = presence of 3.g (MIRCA2000 area of the crop > 0.05 % of the cell land area);
1. `<crop>_<level>_1_ERA5.png`: ERA5 1981–2014;
2. `<crop>_<level>_2_2025_hosing_vs_nohosing.png`: 2025 no hosing, 2025 hosing, land lost and gained with the
   hosing (reference: no hosing);
3. `<crop>_<level>_3_2050_hosing_vs_nohosing.png`: the same for 2050;
4. `<crop>_<level>_4_2025_vs_ERA5.png`: ERA5, land lost and gained from ERA5 in 2025 no hosing and in 2025 hosing;
5. `<crop>_<level>_5_2050_vs_ERA5.png`: the same for 2050;
6. `<crop>_<level>_6_2025_losses_on_today_cropland.png`: what the land where the crop is grown today (MIRCA2000,
   presences of 3.g) loses in 2025, one row per run (no hosing, hosing): today's cropland of the crop, the suitability
   of the run, and on today's cropland the cells still suitable, lost (suitable on ERA5, not in the run), already not
   suitable on ERA5, and with a climate never seen in training in the run; the share of the MIRCA2000 area of the crop
   lost in the title; third row: hosing against no hosing on today's cropland (suitable only with the collapse,
   only without it, in both, lost in both runs to the warming), the effect of the AMOC collapse on the land where the
   crop is grown today. In every panel the cultivated cells that the model does not call suitable on ERA5 are "not
   assessable (model miss)": the model is already wrong there, so nothing is said about their future;
7. `<crop>_<level>_7_2050_losses_on_today_cropland.png`: the same for 2050.

Change maps: lost (red), gained (dark green), suitable in both (light green), not suitable in both (light grey),
climate never seen in training already in the reference (dark grey) and only in the new climate (purple: where the
run leaves the climates the model knows). "Never seen in training" = novel climate of 4.g: unlike the ERA5 1981–2014
climates of today's MIRCA2000 cropland on which the SVM was trained (MESS < 0, NT2 > 1 or no support vector within the
reach of the kernel), so the model cannot assess the suitability there; not a climate new on Earth. The land gained
and lost (Mha, these cells left out) in the title; the definition is written under every figure.

In [ ]:
MAP_LEVELS = ["low", "high"]       # GAEZ input level(s) of the maps
from matplotlib.patches import Patch
NA = "climate never seen in training"   # novel climate of 4.g
CAPTION = ("Training: ERA5 1981-2014 climates of today's MIRCA2000 cropland, described over the growing season of the crop "
           "(GGCMI calendar). \"Climate never seen in training\" = outside them (MESS, NT2, kernel support): the model of "
           "this crop cannot assess the suitability there.")
BIN = (["#e6e6e6", "#1a9641", "#7f7f7f"], ["not suitable", "suitable", NA])
CHG = ["#e6e6e6", "#a6d96a", "#d7191c", "#1a9641", "#7f7f7f", "#9e9ac8"]   # + novel in the reference | novel only in the run
chg_labels = lambda ref, run: ["not suitable in both", "suitable in both", "lost", "gained",
                               f"{ref}: climate never seen in training", f"only {run}: climate never seen in training"]
MIR = (["#e6e6e6", "#1a9641", "#d7191c", "#a6d96a", "#7f7f7f"],
       ["neither", "suitable and cultivated", "cultivated, not suitable", "suitable, not cultivated", NA])
TODAY = (["#e6e6e6", "#1a9641"], ["not cultivated today", "cultivated today (MIRCA2000)"])
LOSS = (["#e6e6e6", "#a6d96a", "#d7191c", "#bdbdbd", "#9e9ac8"],   # model miss in grey: context, not a result
        ["not cultivated today", "still suitable", "lost (suitable on ERA5)", "not assessable: cultivated but not suitable on ERA5 (model miss)"])
# the effect of the collapse (blue, red) in strong colours, the context (suitable in both, model miss, lost in both, novel) pale
AMOC_TODAY = (["#e6e6e6", "#c7e9c0", "#2166ac", "#d7191c", "#bdbdbd", "#d8b365", "#cbc9e2"],
              ["not cultivated today", "suitable with and without the collapse", "suitable only with the collapse (hosing)",
               "suitable only without the collapse (no hosing)", "not assessable: cultivated but not suitable on ERA5 (model miss)",
               "lost with and without the collapse (suitable on ERA5)", "climate never seen in training in either"])
NICE = {"Maize_rf": "maize", "Potato_rf": "potato", "Soybean_rf": "soybean", "Rice_rf": "rice",
        "Wheat_rf": "wheat (winter + spring)", "Wheat-winter_rf": "winter wheat", "Wheat-spring_rf": "spring wheat"}


def panel(ax, classes, colours, labels, title, outside=False):
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, classes, np.nan), vmin=-0.5, vmax=len(colours) - 0.5,
                  cmap=ListedColormap(colours), shading="nearest")
    ax.set_ylim(-60, 85); ax.set_title(title, fontsize=10)
    hs = [Patch(color=c, label=l) for c, l in zip(colours, labels)]
    if outside:                                   # in the empty space left of the map
        ax.legend(handles=hs, loc="center right", bbox_to_anchor=(-0.06, 0.5), fontsize=10, framealpha=0.9)
    else:
        ax.legend(handles=hs, loc="lower left", fontsize=7, framealpha=0.9)


def binary(k, S, N):
    return np.where(N[k], 2.0, S[k].astype(float))


def change(ref, new, S, N):
    # 0-3: not suitable / suitable in both, lost, gained; 4: novel in the reference; 5: novel only in the new climate
    a, b, nov = S[ref], S[new], N[ref] | N[new]
    c = np.where(a & b, 1.0, np.where(a & ~b, 2.0, np.where(~a & b, 3.0, 0.0)))
    c = np.where(N[ref], 4.0, np.where(N[new], 5.0, c))
    g, l = mha(~nov & ~a & b), mha(~nov & a & ~b)
    return c, g, l


def save(fig, path):
    import textwrap
    fig.text(0.5, -0.01, textwrap.fill(CAPTION, int(fig.get_figwidth() * 13)), ha="center", va="top", fontsize=9, style="italic")
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.close(fig)


written = []
for crop in CROPS:
    D = OUT / crop; D.mkdir(exist_ok=True)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * np.nan_to_num(cu.land_area_ha(CFG)))   # presences of 3.g
    for lv in MAP_LEVELS:
        S, N = MAPS[crop]["suit"][lv], MAPS[crop]["novel"]
        head = f"{NICE.get(crop, crop)}, {lv} input (climate >= {cu.CLIMATE_P} and >= TMIN, soil >= {cu.SOIL_MIN})"
        e = SCEN.index("ERA5")
        s0 = S[e] & ~N[e]
        c = np.where(N[e], 4.0, np.where(s0 & pres, 1.0, np.where(pres, 2.0, np.where(s0, 3.0, 0.0))))
        fig, ax = plt.subplots(figsize=(11, 5.2))
        panel(ax, c, *MIR, f"{head}: ERA5 vs MIRCA2000 | {100 * crop_ha[s0].sum() / crop_ha.sum():.0f} % of the crop area and "
              f"{100 * (s0 & pres).sum() / pres.sum():.0f} % of its cells suitable")
        f = D / f"{crop}_{lv}_0_ERA5_vs_MIRCA.png"; save(fig, f); written.append(f)
        fig, ax = plt.subplots(figsize=(11, 5.2))
        panel(ax, binary(e, S, N), *BIN, f"{head}: ERA5 1981-2014 | suitable {mha(S[e]):.0f} Mha")
        save(fig, D / f"{crop}_{lv}_1_ERA5.png"); written.append(D / f"{crop}_{lv}_1_ERA5.png")
        for n, (hos, orig) in enumerate(PAIRS, start=2):
            yr = hos.split("_")[1]
            h, o = SCEN.index(hos), SCEN.index(orig)
            c, g, l = change(o, h, S, N)
            fig, axes = plt.subplots(1, 3, figsize=(30, 5.2))
            panel(axes[0], binary(o, S, N), *BIN, f"{head}: {yr} no hosing | suitable {mha(S[o]):.0f} Mha")
            panel(axes[1], binary(h, S, N), *BIN, f"{head}: {yr} hosing | suitable {mha(S[h]):.0f} Mha")
            panel(axes[2], c, CHG, chg_labels(f"{yr} no hosing", f"{yr} hosing"),
                  f"{yr} hosing vs no hosing: gained {g:.0f} Mha, lost {l:.0f} Mha")
            f = D / f"{crop}_{lv}_{n}_{yr}_hosing_vs_nohosing.png"; save(fig, f); written.append(f)
        for n, (hos, orig) in enumerate(PAIRS, start=4):
            yr = hos.split("_")[1]
            fig, axes = plt.subplots(1, 3, figsize=(30, 5.2))
            panel(axes[0], binary(e, S, N), *BIN, f"{head}: ERA5 1981-2014 | suitable {mha(S[e]):.0f} Mha")
            for ax, (run, lab) in zip(axes[1:], ((orig, "no hosing"), (hos, "hosing"))):
                c, g, l = change(e, SCEN.index(run), S, N)
                panel(ax, c, CHG, chg_labels("ERA5", f"{yr} {lab}"), f"{yr} {lab} vs ERA5: gained {g:.0f} Mha, lost {l:.0f} Mha")
            f = D / f"{crop}_{lv}_{n}_{yr}_vs_ERA5.png"; save(fig, f); written.append(f)
        for n, (hos, orig) in enumerate(PAIRS, start=6):            # losses on today's cropland of the crop
            yr = hos.split("_")[1]
            fig, axes = plt.subplots(3, 3, figsize=(30, 15.6))
            for row, (run, lab) in enumerate(((orig, "no hosing"), (hos, "hosing"))):
                k = SCEN.index(run)
                panel(axes[row, 0], pres.astype(float), *TODAY,
                      f"{NICE.get(crop, crop)}: cultivated today (MIRCA2000: {crop_ha.sum() / 1e6:.1f} Mha harvested in {int(pres.sum())} cells)")
                panel(axes[row, 1], binary(k, S, N), *BIN, f"{head}: {yr} {lab} | suitable {mha(S[k]):.0f} Mha")
                nov_k = pres & N[k]
                miss = pres & ~S[e] & ~nov_k                  # the model misses them on ERA5: nothing to say about their future
                kept, lost = pres & S[e] & S[k], pres & S[e] & ~S[k] & ~nov_k
                c = np.where(nov_k, 4.0, np.where(miss, 3.0, np.where(kept, 1.0, np.where(lost, 2.0, 0.0))))
                share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
                panel(axes[row, 2], c, LOSS[0], LOSS[1] + [f"climate never seen in training ({yr} {lab})"],
                      f"{yr} {lab} on today's cropland: lost {share(lost):.1f} % of the crop area ({mha(lost):.0f} Mha), "
                      f"still suitable {share(kept):.1f} %, not assessable {share(miss):.1f} %")
            h, o = SCEN.index(hos), SCEN.index(orig)
            nov2 = pres & (N[h] | N[o])
            miss = pres & ~S[e] & ~nov2                    # model miss on ERA5: not assessable
            ok = pres & S[e] & ~nov2                       # the cells the model gets right today
            only_h, only_o = ok & S[h] & ~S[o], ok & S[o] & ~S[h]
            both_s, lost_both = ok & S[h] & S[o], ok & ~S[h] & ~S[o]   # lost in both runs: the warming, not the AMOC
            c = np.where(nov2, 6.0, np.where(miss, 4.0, np.where(both_s, 1.0, np.where(only_h, 2.0, np.where(only_o, 3.0,
                np.where(lost_both, 5.0, 0.0))))))
            axes[2, 0].axis("off"); axes[2, 2].axis("off")
            panel(axes[2, 1], c, *AMOC_TODAY,
                  f"{yr} on today's cropland, hosing vs no hosing: suitable only with the collapse {share(only_h):.1f} %, "
                  f"only without it {share(only_o):.1f} % of the crop area (net {share(only_h) - share(only_o):+.1f} points); "
                  f"lost in both {share(lost_both):.1f} %", outside=True)
            f = D / f"{crop}_{lv}_{n}_{yr}_losses_on_today_cropland.png"; save(fig, f); written.append(f)
print(f"{len(written)} maps written in {OUT}/<crop>/")
for f in written[:5]:
    print("  ", f.relative_to(OUT))